# Demonstrating Historical `padding_status_timeseries` Caching in `ngd_proxy`

This notebook demonstrates using the newly implemented **`padding_status_timeseries`** historical caching function. 

This function returns an indicator series showing whether price records were date-padded (e.g. copied from prior close on a holiday). It is fully cached under the hood using SQLite indexing and snappy Parquet disk cache.

### 1. Import and Initialize Cache Manager
We initialize the cache manager using the package config:

In [ ]:
import os
from ngd_proxy import NorgateDataCache

config_path = "config.json"
if not os.path.exists(config_path) and os.path.exists("../config.json"):
    config_path = "../config.json"

ngd_cache = NorgateDataCache(config_path=config_path)
print(f"Cache directory resolved: {ngd_cache.cache_dir}")

### 2. Retrieve Date Padding Status as `pandas-dataframe`
Let's retrieve the padding status history for `TSLA` as a standard Pandas DataFrame:

In [ ]:
df_tsla = ngd_cache.padding_status_timeseries(
    symbol="TSLA",
    timeseriesformat="pandas-dataframe",
    start_date="2025-01-01",
    end_date="2025-01-10"
)

print("DataFrame Shape:", df_tsla.shape)
print("DataFrame Index name:", df_tsla.index.name)
print("DataFrame Columns:", list(df_tsla.columns))
print("\nDataFrame Head:")
print(df_tsla.head())

### 3. Retrieve Date Padding Status as structured `numpy-recarray` (Default)
By default, `padding_status_timeseries` returns structured record arrays (`numpy.recarray`) containing both `Date` and `PaddingStatus` fields:

In [ ]:
rec_msft = ngd_cache.padding_status_timeseries(
    symbol="MSFT",
    timeseriesformat="numpy-recarray",
    start_date="2025-01-01",
    end_date="2025-01-10"
)

print("Recarray type:", type(rec_msft))
print("Recarray Fields:", rec_msft.dtype.names)
print("First 5 rows of recarray:")
for row in rec_msft[:5]:
    print(f"  Date: {row.Date} | Padded: {row.PaddingStatus}")

### 4. Retrieve Date Padding Status as standard `numpy-ndarray`
You can also request a standard 2D NumPy array:

In [ ]:
arr_tsla = ngd_cache.padding_status_timeseries(
    symbol="TSLA",
    timeseriesformat="numpy-ndarray",
    start_date="2025-01-01",
    end_date="2025-01-10"
)

print("Array Type :", type(arr_tsla))
print("Array Shape:", arr_tsla.shape)
print("First 3 elements of NDArray:\n", arr_tsla[:3])

### 5. Direct Convenience Imports and Error Handling
We can also use direct top-level convenience imports from `ngd_proxy`:

In [ ]:
from ngd_proxy import padding_status_timeseries

try:
    # Querying unsupported symbols in Mock Mode raises an exception
    padding_status_timeseries("AAPL")
except Exception as e:
    print(f"Successfully caught expected exception for AAPL in Mock Mode:\n{e}")